In [1]:
from llm_client import LLMClient

llm_client = LLMClient()

OPENROUTER LLM CLIENT
Environment file: C:\Users\User\Nasa Rag SYS\rag\.env
Environment exists: True
API key loaded: True
Primary model: qwen/qwen-2.5-72b-instruct
Fallback model: qwen/qwen3-30b-a3b
Temperature: 0.7
Timeout: 300s


In [2]:
# ============================================================
# INITIALIZE RAG PIPELINE
# ============================================================

from pathlib import Path
import sys

# Notebook:
# rag/Test/Test.ipynb
CURRENT_DIR = Path.cwd()
RAG_DIR = CURRENT_DIR.parent

if str(RAG_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_DIR))


# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------

from retrieval.hybrid_retriever import HybridRetriever
from retrieval.reranker import CrossEncoderReranker
from vector_store.Metadata_filter import MetadataFilter

from context.context_selector import ContextSelector
from context.context_builder import ContextBuilder


# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

CHUNKS_FILE = (
    RAG_DIR
    / "embeddings"
    / "processed"
    / "chunks"
    / "chunks.json"
)


# ------------------------------------------------------------
# INITIALIZE
# ------------------------------------------------------------

print("Initializing Hybrid Retriever...")
hybrid_retriever = HybridRetriever()

print("Initializing Reranker...")
reranker = CrossEncoderReranker()

print("Initializing Metadata Filter...")
metadata_filter = MetadataFilter(
    chunks_file=str(CHUNKS_FILE)
)

print("Initializing Context Selector...")
context_selector = ContextSelector(
    top_k=3
)

print("Initializing Context Builder...")
context_builder = ContextBuilder()


print("\n" + "=" * 70)
print("RAG COMPONENTS READY")
print("=" * 70)

Initializing Hybrid Retriever...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Initializing Reranker...
CROSS-ENCODER RERANKER
Loading model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded successfully.
Initializing Metadata Filter...
Loaded 1200 chunks.
Initializing Context Selector...
Initializing Context Builder...

RAG COMPONENTS READY


In [3]:
# ============================================================
# BUILD REAL CONTEXT
# ============================================================

query = (
    "How does oxygen concentration affect "
    "flame behavior in microgravity?"
)


# Hybrid
hybrid_results = hybrid_retriever.search(
    query=query,
    dense_top_k=20,
    bm25_top_k=20,
    top_k=20
)


# Enrichment
enriched_results = metadata_filter.enrich_results(
    hybrid_results
)


# Reranking
reranked_results = reranker.rerank(
    query=query,
    candidates=enriched_results,
    top_k=2
)


# Selection
selected_results = context_selector.select(
    reranked_results
)


# Context
context = context_builder.build(
    selected_results
)


print("=" * 80)
print("CONTEXT READY")
print("=" * 80)

print(context)


CROSS-ENCODER RERANKING
Query: How does oxygen concentration affect flame behavior in microgravity?
Candidates received: 20
Generating reranker scores...

RERANKING COMPLETE
Top K: 2

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 143
Reranker Score: -2.8965
Hybrid Rank: 14
Dense Rank: None
BM25 Rank: 7

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 191
Reranker Score: -3.0640
Hybrid Rank: 20
Dense Rank: None
BM25 Rank: 10
CONTEXT READY
[EVIDENCE 1]

Experiment ID: EXP-0036
Investigation: FLEX-2
Material: Nomex
Material Type: solid

Experimental Conditions:
- Oxygen concentration: 16.97%
- Pressure: 85.5 kPa
- Flow velocity: 13.93 cm/s
- Ambient temperature: 27.4 °C

Outcome: steady_spread
Risk level: medium

Experimental Evidence:
# Experiment EXP-0036

## Fire Risk

- **Risk Score:** 44.4
- **Risk Level:** medium

## Experimental Summary

Experiment EXP-0036 was conducted as part of the
FLEX-2 investigation using Nomex.


In [4]:
# ============================================================
# TEST PROMPT BUILDER
# ============================================================

from llm.prompt_builder import PromptBuilder

prompt_builder = PromptBuilder()

prompt = prompt_builder.build(
    question=query,
    context=context,
    ml_prediction=None
)

print("=" * 80)
print("PROMPT BUILDER OUTPUT")
print("=" * 80)

print(prompt)

PROMPT BUILDER OUTPUT
You are the scientific reasoning component of the
"Flame in Freefall" microgravity combustion system.

The system retrieved scientific evidence from its knowledge
base.

Your job is to produce a structured scientific response.

USER QUESTION

How does oxygen concentration affect flame behavior in microgravity?

RETRIEVED EXPERIMENTAL EVIDENCE

[EVIDENCE 1]

Experiment ID: EXP-0036
Investigation: FLEX-2
Material: Nomex
Material Type: solid

Experimental Conditions:
- Oxygen concentration: 16.97%
- Pressure: 85.5 kPa
- Flow velocity: 13.93 cm/s
- Ambient temperature: 27.4 °C

Outcome: steady_spread
Risk level: medium

Experimental Evidence:
# Experiment EXP-0036

## Fire Risk

- **Risk Score:** 44.4
- **Risk Level:** medium

## Experimental Summary

Experiment EXP-0036 was conducted as part of the
FLEX-2 investigation using Nomex.

The experiment used an oxygen concentration of
16.97%, a pressure of
85.5 kPa, and a flow velocity of
13.93 cm/s.

The observed flame sp

In [5]:
# ============================================================
# TEST RESPONSE GENERATOR
# ============================================================

from llm.llm_client import LLMClient
from llm.response_generator import ResponseGenerator


llm_client = LLMClient()

response_generator = ResponseGenerator(
    llm_client=llm_client,
    prompt_builder=prompt_builder
)


response = response_generator.generate(
    question=query,
    context=context,
    ml_prediction=None
)


print("=" * 80)
print("FINAL STRUCTURED RESPONSE")
print("=" * 80)

print(response)

OPENROUTER LLM CLIENT
Environment file: C:\Users\User\Nasa Rag SYS\rag\.env
Environment exists: True
API key loaded: True
Primary model: qwen/qwen-2.5-72b-instruct
Fallback model: qwen/qwen3-30b-a3b
Temperature: 0.7
Timeout: 300s

CALLING PRIMARY MODEL
Model: qwen/qwen-2.5-72b-instruct

Primary model succeeded.
FINAL STRUCTURED RESPONSE
{'answer': 'In microgravity, a decrease in oxygen concentration can lead to the extinguishment of flames. At 16.97% oxygen concentration, a steady flame spread was observed, while at 14.32% oxygen concentration, the flame was extinguished.', 'experimental_evidence': [{'experiment_id': 'EXP-0036', 'oxygen_concentration': 16.97, 'flame_temperature': None, 'source': 'FLEX-2 investigation', 'finding': 'Steady flame spread at 16.97% oxygen concentration with a spread rate of 2.746 mm/s and a heat release rate of 1.906 kW.'}, {'experiment_id': 'EXP-0048', 'oxygen_concentration': 14.32, 'flame_temperature': None, 'source': 'FLEX-2 investigation', 'finding': 'F

In [6]:
# ============================================================
# FINAL STRUCTURED RESPONSE
# ============================================================

import json


# ------------------------------------------------------------
# GENERATE RESPONSE
# ------------------------------------------------------------

response = response_generator.generate(
    question=query,
    context=context,
    ml_prediction=None
)


# ------------------------------------------------------------
# DISPLAY STRUCTURED OUTPUT
# ------------------------------------------------------------

print("=" * 80)
print("FINAL STRUCTURED LLM RESPONSE")
print("=" * 80)

print(
    json.dumps(
        response,
        indent=2,
        ensure_ascii=False
    )
)


CALLING PRIMARY MODEL
Model: qwen/qwen-2.5-72b-instruct

Primary model succeeded.
FINAL STRUCTURED LLM RESPONSE
{
  "answer": "In microgravity, the oxygen concentration significantly affects flame behavior. Higher oxygen concentrations support steady flame spread, while lower concentrations lead to flame extinguishment.",
  "experimental_evidence": [
    {
      "experiment_id": "EXP-0036",
      "oxygen_concentration": 16.97,
      "flame_temperature": null,
      "source": "FLEX-2",
      "finding": "At 16.97% oxygen concentration, the flame spread rate was 2.746 mm/s, and the heat release rate was 1.906 kW, resulting in steady flame spread."
    },
    {
      "experiment_id": "EXP-0048",
      "oxygen_concentration": 14.32,
      "flame_temperature": null,
      "source": "FLEX-2",
      "finding": "At 14.32% oxygen concentration, the flame spread rate was 0.0 mm/s, and the heat release rate was 0.493 kW, leading to flame extinguishment."
    }
  ],
  "prediction": null,
  "observ

In [7]:
# ============================================================
# SAVE FINAL RESPONSE AS JSON
# ============================================================

import json
from pathlib import Path


# ------------------------------------------------------------
# GENERATE RESPONSE
# ------------------------------------------------------------

response = response_generator.generate(
    question=query,
    context=context,
    ml_prediction=None
)


# ------------------------------------------------------------
# OUTPUT PATH
# ------------------------------------------------------------

OUTPUT_DIR = Path("outputs")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = (
    OUTPUT_DIR / "final_response.json"
)


# ------------------------------------------------------------
# SAVE JSON
# ------------------------------------------------------------

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        response,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# CONFIRM
# ------------------------------------------------------------

print("=" * 80)
print("RESPONSE SAVED")
print("=" * 80)

print(f"File: {OUTPUT_FILE.resolve()}")
print(f"Size: {OUTPUT_FILE.stat().st_size} bytes")


CALLING PRIMARY MODEL
Model: qwen/qwen-2.5-72b-instruct

Primary model failed:
Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'qwen/qwen-2.5-72b-instruct is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'DeepInfra', 'is_byok': False, 'provider_error_code': 'engine_overloaded', 'limit_source': 'upstream_provider_shared_pool', 'remedy_hint': 'Retry shortly, add your own provider key (https://openrouter.ai/settings/integrations), or route to another provider with provider routing: https://openrouter.ai/docs/features/provider-routing'}}, 'user_id': 'user_3JbETVp0oBMljP2qwqaDGaS6l9m'}

CALLING FALLBACK MODEL
Model: qwen/qwen3-30b-a3b

Fallback model succeeded.
RESPONSE SAVED
File: C:\Users\User\Nasa Rag SYS\rag\llm\outputs\final_response.json
Size: 1964 bytes
